# LAMU 2025 - Porownanie embeddingow: stary model vs PolDense-1B

### Po co ten notatnik?

Oryginalna mapa pytan LAMU 2025 powstala przy uzyciu modelu **`sdadas/st-polish-paraphrase-from-distilroberta`** (768 wymiarow). Ten notatnik **koduje te same pytania dwoma modelami** i pokazuje obok siebie, jak zmienia sie mapa, gdy zamienimy stary model na nowszy **`OPI-PIB/PolDense-1B`** (ModernBERT, 1024 wymiary) z kolekcji PolDense/EuroDense (OPI-PIB).

Dzieki temu widac zarowno **wizualnie** (dwie mapy UMAP obok siebie), jak i **liczbowo** (dwie miary), na ile nowy model inaczej organizuje pytania.

### Jak uruchomic

1. Otworz [Google Colab](https://colab.research.google.com/), wgraj ten plik (.ipynb).
2. **Srodowisko uruchomieniowe > Zmien typ srodowiska > GPU (T4)** - PolDense-1B jest duzy (~1 mld parametrow), na GPU dziala znacznie szybciej. Bez GPU tez zadziala, ale wolniej.
3. **Srodowisko uruchomieniowe > Uruchom wszystko**.

### Co porownujemy

| | Stary model | Nowy model |
|---|---|---|
| Nazwa | `sdadas/st-polish-paraphrase-from-distilroberta` | `OPI-PIB/PolDense-1B` |
| Architektura | DistilRoBERTa | ModernBERT (1B) |
| Wymiar | 768 | 1024 |
| Prefiks | brak | `[sts]: ` (podobienstwo semantyczne) |

**Uwaga o prefiksie:** modele PolDense wymagaja doklejenia prefiksu do tekstu. Dla zadania "jak podobne sa do siebie zdania" (a tym wlasnie jest nasza mapa) uzywamy prefiksu `[sts]: `. Robimy to automatycznie w kodzie.

In [ ]:
!pip install -q -U "sentence-transformers>=5.4.0" transformers umap-learn plotly pymupdf requests scikit-learn

In [ ]:
import fitz  # pymupdf
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
pio.renderers.default = 'colab'

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Urzadzenie: {DEVICE}")

## 1. Pobranie i wczytanie pytan z PDF

(Ten sam krok co w oryginalnym notatniku - pobieramy PDF i wyciagamy pytania z kategoriami.)

In [ ]:
import requests

PDF_URL = "https://radionaukowe.pl/wp-content/uploads/2026/03/LAMU-zadane-pytania-2026.pdf"
PDF_PATH = "LAMU-zadane-pytania-2026.pdf"

response = requests.get(PDF_URL)
response.raise_for_status()
with open(PDF_PATH, "wb") as f:
    f.write(response.content)
print(f"Pobrano PDF: {PDF_PATH} ({len(response.content) / 1024:.0f} KB)")

In [ ]:
def extract_questions_from_pdf(pdf_path):
    """Extract questions grouped by topic from the LAMU PDF."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()

    topic_pattern = r'(FIZYKA|BIOLOGIA|WSZECH[ŚS]WIAT|CZ[ŁL]OWIEK|TECHNOLOGIE|ZIEMIA|MATEMATYKA|HISTORIA|CHEMIA)'
    parts = re.split(topic_pattern, full_text)

    questions = []
    current_topic = None

    for part in parts:
        part_stripped = part.strip()
        if re.match(topic_pattern, part_stripped):
            current_topic = part_stripped.replace('Ś', 'S').replace('Ł', 'L')
            topic_display = {
                'FIZYKA': 'Fizyka', 'BIOLOGIA': 'Biologia', 'WSZECHSWIAT': 'Wszechświat',
                'CZLOWIEK': 'Człowiek', 'TECHNOLOGIE': 'Technologie', 'ZIEMIA': 'Ziemia',
                'MATEMATYKA': 'Matematyka', 'HISTORIA': 'Historia', 'CHEMIA': 'Chemia'
            }
            current_topic = topic_display.get(current_topic, current_topic)
            continue

        if current_topic is None:
            continue

        lines = part.split('\n')
        buffer = ""
        for line in lines:
            line = line.strip()
            if not line or line.startswith('LAMU') or line.startswith('Radio') or 'RADIO' in line.upper() or 'NAUKOWE' in line.upper() or 'LETNIA' in line.upper() or 'AKADEMIA' in line.upper():
                continue
            if re.match(r'^[–\-]\s', line):
                if buffer:
                    questions.append((current_topic, buffer.strip()))
                buffer = re.sub(r'^[–\-]\s*', '', line)
            else:
                if buffer:
                    buffer += ' ' + line

        if buffer:
            questions.append((current_topic, buffer.strip()))

    return questions

questions_data = extract_questions_from_pdf(PDF_PATH)
df = pd.DataFrame(questions_data, columns=['topic', 'question'])
print(f"Liczba pytań: {len(df)}")
print(df['topic'].value_counts())
df.head(5)

## 2. Konfiguracja modeli

Ladujemy dwa modele. PolDense-1B (ModernBERT) probujemy zaladowac z akceleracja `flash_attention_2`, a jesli nie jest dostepna (typowe na Colab T4), automatycznie schodzimy do `sdpa`, a w ostatecznosci `eager`. Na GPU uzywamy `bfloat16`, na CPU standardowego `float32`.

In [ ]:
def load_poldense(model_id, device):
    """Zaladuj model PolDense z rozsadnymi ustawieniami i fallbackiem attn."""
    if device == "cuda":
        for attn in ["flash_attention_2", "sdpa", "eager"]:
            try:
                m = SentenceTransformer(
                    model_id, device=device,
                    model_kwargs={"dtype": "bfloat16", "attn_implementation": attn},
                )
                print(f"  {model_id}: zaladowano (attn_implementation={attn}, bfloat16)")
                return m
            except Exception as e:
                print(f"  {model_id}: attn={attn} nie zadzialalo -> {type(e).__name__}")
    m = SentenceTransformer(model_id, device=device)
    print(f"  {model_id}: zaladowano (domyslne ustawienia, {device})")
    return m


# Konfiguracja: klucz, etykieta na wykresach, funkcja ladujaca, prefiks tekstu
MODELS = {
    "old": {
        "id": "sdadas/st-polish-paraphrase-from-distilroberta",
        "label": "Stary: st-polish-paraphrase-from-distilroberta (768d)",
        "prefix": "",
        "loader": lambda mid: SentenceTransformer(mid, device=DEVICE),
    },
    "poldense": {
        "id": "OPI-PIB/PolDense-1B",
        "label": "Nowy: PolDense-1B (1024d)",
        "prefix": "[sts]: ",
        "loader": lambda mid: load_poldense(mid, DEVICE),
    },
}

for key, cfg in MODELS.items():
    print(f"\n[{key}] laduje {cfg['id']} ...")
    cfg["model"] = cfg["loader"](cfg["id"])
    print(f"  wymiar embeddingow: {cfg['model'].get_sentence_embedding_dimension()}")

## 3. Kodowanie pytan oboma modelami

Kazdy model koduje **te same** pytania. Do PolDense doklejamy prefiks `[sts]: `. Embeddingi normalizujemy (dlugosc 1), zeby porownanie kosinusowe bylo spojne.

In [ ]:
embeddings = {}
questions = df['question'].tolist()

for key, cfg in MODELS.items():
    texts = [cfg["prefix"] + q for q in questions]
    emb = cfg["model"].encode(
        texts, show_progress_bar=True, batch_size=16,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    embeddings[key] = np.asarray(emb, dtype=np.float32)
    print(f"[{key}] embeddingi: {embeddings[key].shape}")

## 4. Redukcja UMAP (osobno dla kazdego modelu)

Ten sam UMAP (`n_neighbors=15, min_dist=0.1, metric=cosine, random_state=42`) uruchamiamy osobno na embeddingach z kazdego modelu. Wspolrzedne zapisujemy jako `x_old/y_old` oraz `x_poldense/y_poldense`.

In [ ]:
reducers = {}  # zachowujemy wytrenowane UMAP-y, zeby pozniej wprojektowac nowe pytania
for key in MODELS:
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric='cosine', random_state=42)
    coords = reducer.fit_transform(embeddings[key])
    df[f'x_{key}'] = coords[:, 0]
    df[f'y_{key}'] = coords[:, 1]
    reducers[key] = reducer
    print(f"[{key}] UMAP: {embeddings[key].shape[1]} -> 2")

## 5. Porownanie wizualne (matplotlib, obok siebie)

In [ ]:
topic_colors = {
    'Fizyka': '#1f77b4', 'Biologia': '#2ca02c', 'Wszechświat': '#9467bd',
    'Człowiek': '#d62728', 'Technologie': '#ff7f0e', 'Ziemia': '#8c564b',
    'Matematyka': '#e377c2', 'Historia': '#7f7f7f', 'Chemia': '#17becf'
}

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, key in zip(axes, MODELS):
    for topic in df['topic'].unique():
        mask = df['topic'] == topic
        ax.scatter(df.loc[mask, f'x_{key}'], df.loc[mask, f'y_{key}'],
                   label=topic, color=topic_colors.get(topic, '#333333'),
                   s=45, alpha=0.7, edgecolors='white', linewidth=0.5)
    ax.set_title(MODELS[key]['label'], fontsize=13)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.grid(True, alpha=0.3)
axes[1].legend(title='Kategoria', fontsize=10, loc='best', framealpha=0.9)
fig.suptitle('LAMU 2025 - te same pytania, dwa modele embeddingow', fontsize=15)
plt.tight_layout()
plt.savefig('compare_poldense_static.png', dpi=150, bbox_inches='tight')
plt.show()
print("Zapisano: compare_poldense_static.png")

## 6. Interaktywna mapa porownawcza (Plotly)

Jeden plik HTML, dwa panele obok siebie. Legenda jest wspolna - klikniecie kategorii ukrywa ja na obu mapach jednoczesnie. Najedz na punkt, aby zobaczyc pytanie.

In [ ]:
df['question_hover'] = df['question'].apply(lambda q: '<br>'.join(
    [q[i:i+70] for i in range(0, len(q), 70)]))

topics_order = list(df['topic'].value_counts().index)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.06,
                    subplot_titles=[MODELS['old']['label'], MODELS['poldense']['label']])

for col, key in enumerate(['old', 'poldense'], start=1):
    for topic in topics_order:
        sub = df[df['topic'] == topic]
        fig.add_trace(go.Scatter(
            x=sub[f'x_{key}'], y=sub[f'y_{key}'], mode='markers', name=topic,
            legendgroup=topic, showlegend=(col == 1),
            marker=dict(size=7, color=topic_colors.get(topic, '#333333'),
                        opacity=0.75, line=dict(width=0.5, color='white')),
            customdata=sub[['topic', 'question_hover']].values,
            hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>',
        ), row=1, col=col)

fig.update_layout(
    title='LAMU 2025 - porownanie modeli embeddingow (stary vs PolDense-1B)',
    legend_title_text='Kategoria', font=dict(size=12), width=1300, height=680,
    plot_bgcolor='white', hoverlabel=dict(font_size=11))
fig.update_xaxes(showgrid=True, gridcolor='lightgray', title_text='UMAP 1')
fig.update_yaxes(showgrid=True, gridcolor='lightgray', title_text='UMAP 2')
fig.show()

HTML_PATH = "lamu_compare_poldense.html"
fig.write_html(HTML_PATH, include_plotlyjs=True, full_html=True)
print(f"Zapisano: {HTML_PATH}")

## 7. Porownanie ilosciowe

Dwie miary, ktore ujmuja liczbowo, **na ile nowy model zmienia mape**:

1. **Silhouette (wg kategorii redakcyjnych)** - liczona na pelnych embeddingach z metryka kosinusowa. Mowi, jak dobrze dany model **oddziela kategorie nadane w PDF** (Fizyka, Biologia, ...). Wyzej = kategorie sa bardziej rozdzielone w przestrzeni tego modelu. Wartosci sa zwykle niskie, bo pytania dzieci sa z natury interdyscyplinarne - wazna jest **roznica** miedzy modelami, nie sama wartosc.

2. **Pokrycie sasiadow (top-10)** - dla kazdego pytania bierzemy 10 najblizszych pytan wg starego i wg nowego modelu, i liczymy, jaka czesc sie pokrywa. Sredni wynik ~1.0 = modele "widza" te same sasiedztwa; blizej 0 = nowy model radykalnie przeorganizowal podobienstwa.

In [ ]:
# 1. Silhouette wg kategorii
labels = df['topic'].values
print("Silhouette (metric=cosine, etykiety = kategorie z PDF):")
sil = {}
for key in MODELS:
    sil[key] = silhouette_score(embeddings[key], labels, metric='cosine')
    print(f"  {MODELS[key]['label']:52s} -> {sil[key]:+.4f}")
delta = sil['poldense'] - sil['old']
print(f"\n  Roznica (PolDense - stary): {delta:+.4f}  "
      f"({'PolDense lepiej oddziela kategorie' if delta > 0 else 'stary model lepiej oddziela kategorie'})")

# 2. Pokrycie sasiadow top-k
def mean_neighbor_overlap(emb_a, emb_b, k=10):
    sa = cosine_similarity(emb_a); np.fill_diagonal(sa, -np.inf)
    sb = cosine_similarity(emb_b); np.fill_diagonal(sb, -np.inf)
    na = np.argsort(-sa, axis=1)[:, :k]
    nb = np.argsort(-sb, axis=1)[:, :k]
    return np.mean([len(set(na[i]) & set(nb[i])) / k for i in range(len(emb_a))])

overlap = mean_neighbor_overlap(embeddings['old'], embeddings['poldense'], k=10)
print(f"\nSrednie pokrycie 10 najblizszych sasiadow (stary vs PolDense): {overlap:.3f}")
print(f"  -> srednio {overlap*10:.1f} z 10 najblizszych pytan to te same w obu modelach")

### Jak czytac wyniki

- **Silhouette** rosnie po zmianie na PolDense-1B -> nowy model **lepiej grupuje** pytania zgodnie z kategoriami redakcyjnymi (klastry bardziej rozdzielone). Spada -> stary model lepiej pasowal do tego konkretnego podzialu.
- **Pokrycie sasiadow** blisko 1.0 -> mapy sa podobne, zmiana modelu niewiele zmienia w praktyce. Wartosc niska (np. < 0.5) -> nowy model istotnie przeorganizowal to, co uznaje za "podobne pytania", i warto obejrzec obie mapy z Kroku 6.

Osie UMAP nie maja znaczenia fizycznego, a sam uklad moze byc obrocony/odbity miedzy modelami - dlatego **nie porownujemy pozycji punktow**, tylko wzajemne sasiedztwa i rozdzielenie kolorow.

## 8. Eksport plikow

In [ ]:
for path in ["lamu_compare_poldense.html", "compare_poldense_static.png"]:
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print(f"Otworz lokalnie: {path}")

## 9. Nowe pytania LAMU 2026 - projekcja na wytrenowany UMAP

Tu dodajemy pytania z **planu LAMU 2026** i **projektujemy je na juz wytrenowany UMAP** kazdego modelu (`reducer.transform()`, a NIE `fit_transform`). Dzieki temu nowe punkty laduja w tej samej przestrzeni co pytania z 2025 i widac, obok ktorych tematow sie znajduja - w obu modelach naraz.

**Jak dodawac wlasne pytania:** edytuj slownik `new_questions_2026` ponizej - kazda sekcja to lista pytan. Nazwiska i wiek dzieci oraz linie "Odpowiada prof. ..." pomijamy, zostawiamy samo pytanie.

In [ ]:
# --- Plan LAMU 2026: edytuj/dodawaj pytania tutaj ---
new_questions_2026 = {
    "Bakterie": [
        "Co było pierwsze, bakterie czy wirusy?",
        "Skąd bakterie wiedzą, co mają robić?",
        "Czy we łzach są bakterie?",
        "Skoro antybiotyki zabijają bakterie, to czemu nie potrafią zabić wirusów?",
        "Skoro bakterie są, a ich nie widać, to może krasnoludki też są?",
        "Jaka jest najmniejsza komórka na świecie?",
    ],
    "Gnicie, pleśnienie i kupa": [
        "Dlaczego owoce i warzywa gniją?",
        "Jak jedzenie gnije?",
        "Dlaczego niektóre rzeczy pleśnieją, a inne nie?",
        "Dlaczego rzeczy w lodówce pleśnieją wolniej?",
        "Czemu jabłko staje się brązowe po miesiącu, dwóch w chłodnej temperaturze?",
        "Dlaczemu kupy psa zmieniają kolor po jakimś czasie na biały?",
    ],
    "Muchy i kowale": [
        "Czy muchy mają tak jak my organy wewnętrzne, na przykład jak wątroba, mózg i serce?",
        "Z czego się składa organizm muchy?",
        "Dlaczego mucha lata w tak chaotyczny sposób? Czy ma wyznaczoną jakąś trajektorię?",
        "Czemu muchy muszą mieć krew, żeby złożyć jaja?",
        "Dlaczego kowale bezskrzydłe się łączą?",
    ],
}

new_rows = [(sec, q) for sec, qs in new_questions_2026.items() for q in qs]
df_new = pd.DataFrame(new_rows, columns=['section', 'question'])
print(f"Nowych pytań 2026: {len(df_new)} (sekcje: {df_new['section'].nunique()})")
df_new

In [ ]:
# Embedding nowych pytan kazdym modelem + projekcja na wytrenowany UMAP (transform)
for key, cfg in MODELS.items():
    texts = [cfg['prefix'] + q for q in df_new['question']]
    emb = cfg['model'].encode(texts, show_progress_bar=False, batch_size=16,
                              convert_to_numpy=True, normalize_embeddings=True)
    coords = reducers[key].transform(np.asarray(emb, dtype=np.float32))
    df_new[f'x_{key}'] = coords[:, 0]
    df_new[f'y_{key}'] = coords[:, 1]
    print(f"[{key}] wprojektowano {len(df_new)} nowych pytań na wytrenowany UMAP")

In [ ]:
# Wykres: pytania 2025 (przygaszone, wg kategorii) + pytania 2026 (gwiazdki wg sekcji)
section_colors = {'Bakterie': '#e6194B', 'Gnicie, pleśnienie i kupa': '#3cb44b',
                  'Muchy i kowale': '#4363d8'}
extra = ['#f58231', '#911eb4', '#42d4f4', '#bfef45', '#f032e6']  # dla dodatkowych sekcji

df_new['q_hover'] = df_new['question'].apply(lambda q: '<br>'.join(
    [q[i:i+70] for i in range(0, len(q), 70)]))

fig3 = make_subplots(rows=1, cols=2, horizontal_spacing=0.06,
                     subplot_titles=[MODELS['old']['label'], MODELS['poldense']['label']])

for col, key in enumerate(['old', 'poldense'], start=1):
    # tlo: pytania 2025 wg kategorii, przygaszone
    for topic in topics_order:
        sub = df[df['topic'] == topic]
        fig3.add_trace(go.Scatter(
            x=sub[f'x_{key}'], y=sub[f'y_{key}'], mode='markers', name=topic,
            legendgroup=topic, showlegend=(col == 1),
            marker=dict(size=6, color=topic_colors.get(topic, '#333333'),
                        opacity=0.30, line=dict(width=0)),
            customdata=sub[['topic', 'question_hover']].values,
            hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>',
        ), row=1, col=col)
    # nowe pytania 2026: gwiazdki wg sekcji
    for i, sec in enumerate(new_questions_2026):
        sub = df_new[df_new['section'] == sec]
        fig3.add_trace(go.Scatter(
            x=sub[f'x_{key}'], y=sub[f'y_{key}'], mode='markers',
            name=f'2026: {sec}', legendgroup=f'2026: {sec}', showlegend=(col == 1),
            marker=dict(size=15, symbol='star',
                        color=section_colors.get(sec, extra[i % len(extra)]),
                        opacity=1.0, line=dict(width=1.2, color='black')),
            customdata=sub[['section', 'q_hover']].values,
            hovertemplate='<b>2026 · %{customdata[0]}</b><br>%{customdata[1]}<extra></extra>',
        ), row=1, col=col)

fig3.update_layout(
    title='LAMU 2026 (gwiazdki) wprojektowane na mape 2025 - stary model vs PolDense-1B',
    legend_title_text='Legenda', font=dict(size=12), width=1300, height=700,
    plot_bgcolor='white', hoverlabel=dict(font_size=11))
fig3.update_xaxes(showgrid=True, gridcolor='lightgray', title_text='UMAP 1')
fig3.update_yaxes(showgrid=True, gridcolor='lightgray', title_text='UMAP 2')
fig3.show()

HTML_2026 = "lamu2026_projection_compare.html"
fig3.write_html(HTML_2026, include_plotlyjs=True, full_html=True)
print(f"Zapisano: {HTML_2026}")

try:
    from google.colab import files
    files.download(HTML_2026)
except ImportError:
    print(f"Otworz lokalnie: {HTML_2026}")